# Introduction

We are curious what fraction of a library is filled with spikes?

In [ ]:
import os
from glob import glob
import numpy
import sys
import pandas

In [ ]:
%matplotlib inline

In [ ]:
ROOT=os.path.expanduser("~diane/proj/encode3-curation")
if ROOT not in sys.path:
    sys.path.append(ROOT)

In [ ]:
experiment_qc = pandas.read_csv(os.path.join(ROOT, 'compare-mad', 'experiment-mad-qc.csv'))

In [ ]:
wold = glob(os.path.expanduser('~/public_html/compare-mad/wold_gene_quantifications/*_tpms.csv'))
len(wold)

In [ ]:
gingeras = glob(os.path.expanduser('~/public_html/compare-mad/gingeras_fpkms/*_tpms.csv'))
len(gingeras)

In [ ]:
def make_library_gene_median(experiments):
    for pathname in experiments:
        path, filename = os.path.split(pathname)
        accession = filename.split('_')[0]
        data = pandas.read_csv(pathname, index_col=0, header=0)
        logdata = numpy.log2(data)
        A = logdata.sum(axis=1) / 2.0
        
        gencode_rows = [ x.startswith('ENS') for x in data.index]
        gene_median = data[gencode_rows][A > 0].median()
        for name in gene_median.index:
            yield (accession, name, gene_median[name])

In [ ]:
gingeras_median = pandas.DataFrame(make_library_gene_median(gingeras), columns=['experiment', 'library', 'median'])
gingeras_median.to_csv(os.path.expanduser('~/public_html/compare-mad/gingeras_fpkms/gingeras-median.csv'),
                 index=False)

In [ ]:
gingeras_median

# Gingeras Median

In [ ]:
gingeras_median['median'].plot(kind='hist')

In [ ]:
wold_median = pandas.DataFrame(make_library_gene_median(wold), columns=['experiment', 'library', 'median'])
wold_median.to_csv(os.path.expanduser('~/public_html/compare-mad/wold_gene_quantifications/wold-median.csv'),
                 index=False)

# Wold Median

In [ ]:
wold_median['median'].plot(kind='hist')

# Gingeras MAD vs median

In [ ]:
_ = pandas.merge(
    experiment_qc[['experiment', 'MAD']], 
    gingeras_median[['experiment', 'median']],
    on='experiment', how='inner').plot('MAD', 'median', kind='scatter')

# Wold MAD vs median

In [ ]:
_ = pandas.merge(
    experiment_qc[['experiment', 'MAD']], 
    wold_median[['experiment', 'median']],
    on='experiment', how='inner').plot('MAD', 'median', kind='scatter')